In [ ]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

sys.path.insert(0, str(Path.cwd() / "CLI-Demos"))
sys.path.insert(0, str(Path.cwd() / "CLI-Demos"/"medsam2_function"))

/home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus/CLI-Demos
/home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [ ]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'mp4', 'nifti']


In [ ]:
import pandas as pd

from extract_finetune_dataset import find_bmode_file, find_ceus_file, find_voi_file

MASTER_CSV = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/Motion Compensation Comparison 4 patients(MedSAM2).csv"

df = pd.read_csv(MASTER_CSV)
row = df.iloc[6]

case_key = f"{row['Site']}-{row['Patient Number']}-{row['Visit']}-{row['Bolus']}"
print(f"\n=== {case_key} ===")

try:
    bmode_path = find_bmode_file(row["Data Dir"], row["Visit"], row["Bolus"])
    ceus_path  = find_ceus_file(bmode_path)
    seg_path_  = find_voi_file(row["Data Dir"], row["Visit"], row["Bolus"])
    inputs = {"bmode": bmode_path, "ceus": ceus_path, "seg": seg_path_}
    print(f"  bmode: {os.path.basename(inputs['bmode'])}")
    print(f"  ceus : {os.path.basename(inputs['ceus'])}")
    print(f"  seg  : {os.path.basename(inputs['seg'])}")
except FileNotFoundError as e:
    inputs = {"bmode": None, "ceus": None, "seg": None}
    print(f"  [SKIP] {e}")



=== UCSD-P07-V01-CE1RUN2 ===
  bmode: UCSD-P07-V01-CE1RUN2_12.57.27_mf_sip_capture_50_2_1_0_BMODE.nii
  ceus : UCSD-P07-V01-CE1RUN2_12.57.27_mf_sip_capture_50_2_1_0_CEUS.nii
  seg  : UCSD-P07-V01-CE1RUN2-MC_VOI.nii.gz


In [ ]:
df

,Site,Patient Number,Visit,Bolus,Data Dir,TIC curve save path,nomc_AUC,nomc_PE,nomc_TP,nomc_MTT,...,sam2_PE,sam2_TP,sam2_MTT,sam2_T0,sam2_Mu,sam2_Sigma,sam2_R2,sam2_roughness,sam2_snr,sam2_volume_mm3
0,UCSD,P05,V01,CE1,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.347719e+06,8327.633158,47.536914,383.432851,...,8975.495891,52.800761,363.818451,3.789723e+00,5.253279,1.134351,0.966026,202.171913,205.712944,11964.951175
1,UCSD,P05,V01,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.692861e+06,9435.872726,51.554988,369.244355,...,9978.216332,57.343038,350.113502,1.506493e+00,5.255189,1.098243,0.943186,191.578204,110.398935,12876.443770
2,UCSD,P05,V02,CE1,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,1.712319e+06,6166.061616,40.766837,421.194098,...,5768.764736,73.810720,313.022869,7.473880e-01,5.264685,0.981418,0.870615,296.424250,50.886404,9688.391630
3,UCSD,P05,V02,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,1.545486e+06,5355.496610,58.746114,344.917507,...,4475.074631,69.547323,317.003898,6.665360e-11,5.253279,1.005620,0.878154,196.537790,41.650516,7650.166766
4,UCSD,P05,V03,CE1,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.199110e+06,7795.389946,46.169041,392.423915,...,8458.887316,42.558933,408.729072,2.770115e+00,5.258998,1.228051,0.927537,201.908952,48.378266,6361.262661
5,UCSD,P05,V03,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.186378e+06,7672.627407,46.799319,395.343219,...,8463.288033,40.581516,424.551058,2.700113e+00,5.268459,1.251058,0.944508,187.403573,62.164648,8656.066537
6,UCSD,P07,V01,CE1RUN2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.516931e+06,9404.462362,30.497481,509.045741,...,9832.165646,34.977233,475.330919,1.238431e+01,5.294240,1.318917,0.969462,223.521771,99.747365,3925.238109
7,UCSD,P07,V01,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.906401e+06,10325.053000,49.860917,369.317238,...,10977.000964,49.908712,369.140355,9.269303e+00,5.244183,1.154984,0.937301,181.017471,62.752625,4601.932955
8,UCSD,P07,V02,CE01RUN2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,1.806230e+06,7175.955174,27.168558,504.604722,...,7742.182951,26.977668,372.326067,6.229600e-01,5.044850,1.322815,0.960951,188.531246,269.389169,4005.526408
9,UCSD,P07,V02,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.706491e+06,10040.504852,37.079784,431.932617,...,10452.629547,39.093571,420.660688,3.157327e-11,5.249870,1.258536,0.890329,211.985753,311.489425,5074.992779


In [ ]:
scan_type = 'nifti'

# Takes the NIfTI files as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = inputs['ceus']
bmode_scan_path = inputs['bmode']
scan_loader_kwargs = {
}

In [ ]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [ ]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [ ]:
seg_type = 'nifti'

seg_path = inputs['seg']
seg_loader_kwargs = {}

In [ ]:
from src.entrypoints import seg_loading_step

# Testing the motion compensation, right now is hard coded
seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

## Extract one B-mode frame for fine-tuning annotation

Save a single timepoint from `bmode_scan_path` as `image.nii.gz` + `image.json` (plus an empty `segmentations/` folder), in the layout used for MedSAM2 fine-tuning data prep. Pick `frame_idx` after reviewing the volume in the 2D/3-plane or napari viewer above — a frame where the tumor and tracked bbox look well-conditioned.

In [ ]:
from extract_finetune_dataset import extract_bmode_and_ceus_frame

print(f"BMODE: {bmode_scan_path}")
print(f"CEUS:  {CEUS_scan_path}")
print(f"  {bmode_image_data.pixel_data.shape[-1]} frames available")

# ── Set these for the case you're extracting ─────────────────────────────────
site = bmode_scan_path.split('/')[-1].split('-')[0]
patient =  bmode_scan_path.split('/')[-1].split('-')[1]
visits =  bmode_scan_path.split('/')[-1].split('-')[2]

native_subject_id = site+'-'+patient+'-'+visits
frame_idx         = 0                              # <- pick after reviewing above
# subject_id        = "subject_000001"                # <- one folder per patient
# sequence_id       = "sequence_001"                   # <- based on the bolus number
# output_root       = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_finetune_data"

# out_dir = extract_bmode_and_ceus_frame(
#     bmode_path=bmode_scan_path,
#     ceus_path=CEUS_scan_path,
#     frame_idx=frame_idx,
#     subject_id=subject_id,
#     sequence_id=sequence_id,
#     output_root=output_root,
#     native_subject_id=native_subject_id,
# )
print(f"Saved frame {frame_idx} (image.* + image_ceus.*) -> {out_dir}")

## Batch-generate volumes from manifest.csv

Reads `manifest.csv` (produced by `extract_finetune_dataset.py manifest` from the motion-compensation CSV) and, for every row that has a reviewed `frame_idx` filled in, extracts that frame from both the B-mode and CEUS volumes. No intensity normalization is applied — the saved B-mode volume is the raw pixel data; use ITK-SNAP's own contrast/window-level controls when viewing or annotating it.

In [ ]:
from extract_finetune_dataset import extract_all

manifest_path = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_finetune_data/manifest.csv"
output_root   = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_finetune_data"

extract_all(manifest_path, output_root)